# Objetivo e Método Aplicado

- Objetivo:
O objetivo desse projeto é classificar imagens de plantas de trigo que estão divididas em 3 classes: saudável, ferrugem do tronco e ferrugem das folhas.

- Método:
O método utilizado foi Convolutional Neural Network, especificamente o modelo Inception V3,\
com Transfer Learning.

# Imports

In [ ]:
from pathlib import Path
from imutils import paths

import os

import pickle
import random
import sklearn
import matplotlib
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
import cv2

import tensorflow as tf
import keras
from keras.models import load_model, save_model
from keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.utils import img_to_array, load_img
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

In [ ]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
plt.rcParams['figure.figsize'] = (8,7)

In [ ]:
caminho_dados_treino = f"D:/data/images/deteccao-doencas-plantas/treino"

In [ ]:
p = Path(caminho_dados_treino)
p

In [ ]:
# Lista de caminhos de cada imagem dos dados de Treino

caminho_imagens_treino = sorted(list(paths.list_images(caminho_dados_treino)))

In [ ]:
caminho_imagens_treino[0]

In [ ]:
random.shuffle(caminho_imagens_treino)

In [ ]:
caminho_imagens_treino[:5]

In [ ]:
Path(caminho_imagens_treino[0]).parent.name

# Pre-Processing

In [ ]:
img_1 = cv2.imread(filename=caminho_imagens_treino[0])
np.array(img_1).shape

In [ ]:
# Imagens terão por padrão a dimensão (224,224,3)

img_dim = (224, 224, 3)

In [ ]:
# Loop pelos caminhos da imagens de treino
# Lê cada caminho, lê cada imagem com o cv2.imread(caminho)
# Extrai o label do caminho da imagem

dados = []
labels = []
count = 0

for caminho in caminho_imagens_treino:    
    # Lê a imagem a partir do caminho especificado
    img = cv2.imread(caminho)
    
    # Redimensiona a imagem de acordo com a dim especificada
    img_resized = cv2.resize(img, (img_dim[1], img_dim[0]), cv2.INTER_AREA)
    
    # Usa a função do TF para converter a imagem em numpy array
    array = img_to_array(img_resized)
    dados.append(array)
    
    # Extrai a label
    label = Path(caminho).parent.name
    labels.append(label)
    count += 1

In [ ]:
count

In [ ]:
np.max(array), np.min(array)

In [ ]:
novos_dados = np.array(dados, dtype='float')
X = keras.applications.inception_v3.preprocess_input(novos_dados)

In [ ]:
# São 562 (imagens) arrays
# Cada array tem 3 matrizes, 224W, 224H

X.shape

# Exibe o tamanho da memória que o array ocupa

In [ ]:
# Exibe o tamanho da memória RAM que o ndarray de imagens ocupa.
# Para um resultado em mebibytes, divide por (1024 ** 2)
# MiB é o sistema binário, que é o que realmente o computador usa
# 1 MiB = 1.048.576 bytes

f"O tamanho da memória utilizada pelo array de imagens de treino é: {X.nbytes / (1024 **2):.2f} MiB"

# Binarização dos Labels

In [ ]:
labels = np.array(labels)
labels[:5]

In [ ]:
labels[:3]

In [ ]:
# É chamado de Binarizador, porque ele cria uma coluna para cada variável categórica,
# que pode assumir apenas os valores 0 e 1.
# ou seja, é um one-hot-encoder para labels

lb = LabelBinarizer()

In [ ]:
binarized_labels = lb.fit_transform(labels)
binarized_labels

In [ ]:
binarized_labels.shape

In [ ]:
lb.classes_

In [ ]:
classes = []
for i in lb.classes_:
    classes.append(str(i))
classes

# Salvar o binarizador para usar no deploy

In [ ]:
lb_file = open("./models/label_binarizer.pickle", "wb")
pickle.dump(lb.classes_, lb_file)

In [ ]:
lb_load = open("./models/label_binarizer.pickle", "rb")
lb_loaded = pickle.load(lb_load)

In [ ]:
lb_loaded

# Divisão dos dados de Treino

In [ ]:
X.shape

In [ ]:
binarized_labels.shape

In [ ]:
X_treino, X_valid, y_treino, y_valid = train_test_split(X, binarized_labels, test_size = 0.2)

In [ ]:
X_treino.shape, X_valid.shape

# Modelo CNN

## Construção do Modelo

In [ ]:
img_dim

In [ ]:
len(lb_loaded)

In [ ]:
def cria_modelo(imagem_dims, label_binarizer):

    modelo_base = keras.applications.InceptionV3(
        include_top = False,
        weights = 'imagenet',
        input_shape = imagem_dims,
        classifier_activation='softmax')
    
    modelo_base.trainable = False
    
    novo_modelo = keras.layers.GlobalAveragePooling2D()(modelo_base.output)
    
    modelo_com_transfer_learning = keras.layers.Dense(units = len(label_binarizer),
                                                      activation = 'softmax')(novo_modelo)
    
    modelo_final = keras.Model(inputs = modelo_base.inputs, outputs = modelo_com_transfer_learning)
    
    return modelo_final

In [ ]:
modelo_final = cria_modelo(img_dim, lb_loaded)

In [ ]:
#modelo_final.summary()

## Inspecionar as layers

In [ ]:
modelo_final.layers[0].trainable_weights, modelo_final.layers[0].trainable

In [ ]:
modelo_final.layers[1].trainable_weights

In [ ]:
modelo_final.layers[1].trainable

In [ ]:
# Verifica o total de layers

count_layers = 0
for layer in modelo_final.layers:
    count_layers+=1
        
count_layers

In [ ]:
# Verifica quantas camadas são treináveis

count_trainable_layers = 0
for layer in modelo_final.layers:
    if layer.trainable == True:
        count_trainable_layers +=1

count_trainable_layers

In [ ]:
momentum_list = []

for layer in modelo_final.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        momentum_list.append(layer.momentum)

set(momentum_list)

In [ ]:
# Diminuir o momentum de 0.99 para 0.7

for layer in modelo_final.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.momentum = 0.7

## Checkpoint e Compilação

In [ ]:
caminho_modelo = "./modelos/modelo.keras"

In [ ]:
checkpoint = ModelCheckpoint(filepath = caminho_modelo,
                             monitor = 'val_loss',
                             save_best_only=True,
                             mode='min')

In [ ]:
callback = EarlyStopping(monitor = 'val_loss',
                         mode = 'min',
                         patience = 8)

In [ ]:
callback_list = [checkpoint, callback]

In [ ]:
# Hiper-parâmetros

epochs = 25
learning_rate = 1e-3
batch_size = 32

In [ ]:
modelo_final.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.BinaryAccuracy()],
)

## Treinamento do Modelo

In [ ]:
history = modelo_final.fit(x = X_treino, y = y_treino,
                           batch_size = batch_size,
                           epochs = epochs,
                           validation_data = [X_valid, y_valid],
                           verbose=1,
                           callbacks = callback_list
                           )

# Avaliação do Modelo

In [ ]:
modelo = load_model("./modelos/modelo.keras")

In [ ]:
obj_bin = pickle.load(open("./modelos/label_binarizer.pickle", 'rb'))

In [ ]:
obj_bin = [str(x)for x in obj_bin]
obj_bin

In [ ]:
y_pred_valid = modelo.predict(X_valid)

In [ ]:
y_pred_classes = np.argmax(y_pred_valid, axis = 1)
y_pred_classes[:10]

In [ ]:
y_valid_classes = np.argmax(y_valid, axis = 1)
y_valid_classes[:10]

In [ ]:
accuracy_score(y_pred_classes,y_valid_classes)

# Uso do Modelo com Imagens de Teste

In [ ]:
caminho_dados_teste = Path(r"D:\data\images\deteccao-doencas-plantas\teste")

In [ ]:
caminho_imagens_teste = list(paths.list_images(caminho_dados_teste))

In [ ]:
def read_image(path, modelo, obj_bin):
    random_img = random.choice(path)
    original_img = cv2.imread(random_img)
    
    # Imagem que será exibida
    imagem_copia = original_img.copy()
    h, w = imagem_copia.shape[:2]
    
    # Transforma a imagem e modificada o shape para o padrão Keras
    img = cv2.resize(original_img, (224,224)) / 255.0
    img = img_to_array(img)
    img = np.expand_dims(img, axis = 0) # coloca um novo eixo na posicao 0 (batch) apesar de ser apenas 1 imagem
    
    # Retorna as probabilidades da imagem pertencer a cada classe
    proba = modelo.predict(img)
    
    # Prevê a classe da imagem
    cls = np.argmax(proba)
    
    # Atribui o label
    label = obj_bin[cls]
    
    # Adiciona o texto (label) à imagem
    cv2.putText(img = imagem_copia,
                text = label,
                org = (int(w * 0.05), int(h * 0.08)), # (w,h)
                fontFace=cv2.FONT_HERSHEY_COMPLEX,
                fontScale = h / 700,
                color = (0,0,0), # black
                thickness = max(1, int(h / 300))
    )
    
    plt.imshow(cv2.cvtColor(imagem_copia, cv2.COLOR_BGR2RGB))
    return None

In [ ]:
read_image(caminho_imagens_teste, modelo, obj_bin)

In [ ]:
read_image(caminho_imagens_teste, modelo, obj_bin)

# Fim